In [2]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(r"C:\data\Synthea\csv\patients.csv")   # your path here

df = pd.read_csv(DATA_PATH)
print(df.shape)       # (rows, columns)
df.head()

(1163, 25)


,Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,LAST,SUFFIX,...,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE
0,b9c610cd-28a6-4636-ccb6-c7a0d2a4cb85,2019-02-17,NaN,999-65-3251,NaN,NaN,NaN,Damon455,Langosh790,NaN,...,Middleborough Massachusetts US,620 Lynch Tunnel Apt 0,Springfield,Massachusetts,Hampden County,1104.0,42.080389,-72.480431,9039.1645,7964.1255
1,c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8,2005-07-04,NaN,999-49-3323,S99941126,NaN,NaN,Thi53,Wunsch504,NaN,...,Danvers Massachusetts US,972 Tillman Branch Suite 48,Bellingham,Massachusetts,Norfolk County,NaN,42.035213,-71.482519,402723.4150,14064.1350
2,339144f8-50e1-633e-a013-f361391c4cff,1998-05-11,NaN,999-10-8743,S99996708,X75063318X,Mr.,Chi716,Greenfelder433,NaN,...,Athens Athens Prefecture GR,1060 Bernhard Crossroad Suite 15,Boston,Massachusetts,Suffolk County,2131.0,42.292557,-71.061160,571935.8725,787.5375
3,d488232e-bf14-4bed-08c0-a82f34b6a197,2003-01-28,NaN,999-56-6057,S99929424,NaN,Ms.,Phillis443,Walter473,NaN,...,Boston Massachusetts US,677 Ritchie Terrace,Hingham,Massachusetts,Plymouth County,2043.0,42.200491,-70.916076,582557.8030,104782.2070
4,217f95a3-4e10-bd5d-fb67-0cfb5e8ba075,1993-12-23,NaN,999-91-4320,S99991143,X44132498X,Mr.,Jerrold404,Herzog843,NaN,...,Boston Massachusetts US,276 Bernier Branch,Revere,Massachusetts,Suffolk County,NaN,42.381875,-70.999286,475826.8550,18067.0950


In [3]:
df.dtypes

Id                         str
BIRTHDATE                  str
DEATHDATE                  str
SSN                        str
DRIVERS                    str
PASSPORT                   str
PREFIX                     str
FIRST                      str
LAST                       str
SUFFIX                     str
MAIDEN                     str
MARITAL                    str
RACE                       str
ETHNICITY                  str
GENDER                     str
BIRTHPLACE                 str
ADDRESS                    str
CITY                       str
STATE                      str
COUNTY                     str
ZIP                    float64
LAT                    float64
LON                    float64
HEALTHCARE_EXPENSES    float64
HEALTHCARE_COVERAGE    float64
dtype: object

In [4]:
df.isna().sum().sort_values(ascending=False)

SUFFIX                 1147
DEATHDATE              1000
MAIDEN                  832
ZIP                     545
MARITAL                 384
PASSPORT                276
PREFIX                  245
DRIVERS                 215
Id                        0
LAST                      0
FIRST                     0
BIRTHDATE                 0
SSN                       0
ETHNICITY                 0
GENDER                    0
BIRTHPLACE                0
RACE                      0
ADDRESS                   0
CITY                      0
STATE                     0
COUNTY                    0
LAT                       0
LON                       0
HEALTHCARE_EXPENSES       0
HEALTHCARE_COVERAGE       0
dtype: int64

In [6]:
print("Duplicate IDs:", df["Id"].duplicated().sum())

Duplicate IDs: 0


In [7]:
clean = df.copy()

# 1. Consistent column names: lowercase snake_case
clean.columns = clean.columns.str.strip().str.lower()

# 2. Drop PII you don't need for analytics
pii_cols = ["ssn", "drivers", "passport", "address", "prefix", "suffix", "maiden"]
clean = clean.drop(columns=[c for c in pii_cols if c in clean.columns])

# 3. Text dates -> real dates (bad values become NaT instead of crashing)
clean["birthdate"] = pd.to_datetime(clean["birthdate"], errors="coerce")
clean["deathdate"] = pd.to_datetime(clean["deathdate"], errors="coerce")

# 4. Nulls with meaning become explicit columns or labels
clean["is_deceased"] = clean["deathdate"].notna()
if "marital" in clean.columns:
    clean["marital"] = clean["marital"].fillna("Unknown")

# 5. Trim stray spaces in text columns
text_cols = clean.select_dtypes(include="object").columns
clean[text_cols] = clean[text_cols].apply(lambda s: s.str.strip())

# 6. Remove duplicate patients
clean = clean.drop_duplicates(subset="id")

clean.dtypes

C:\Users\tsais\AppData\Local\Temp\ipykernel_14944\718310754.py:20: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = clean.select_dtypes(include="object").columns


id                                str
birthdate              datetime64[us]
deathdate              datetime64[us]
first                             str
last                              str
marital                           str
race                              str
ethnicity                         str
gender                            str
birthplace                        str
city                              str
state                             str
county                            str
zip                           float64
lat                           float64
lon                           float64
healthcare_expenses           float64
healthcare_coverage           float64
is_deceased                      bool
dtype: object

In [8]:
today = pd.Timestamp.today()

checks = {
    "rows": len(clean),
    "null birthdates": clean["birthdate"].isna().sum(),
    "birthdates in the future": (clean["birthdate"] > today).sum(),
    "death before birth": (clean["deathdate"] < clean["birthdate"]).sum(),
    "duplicate ids": clean["id"].duplicated().sum(),
    "deceased patients": clean["is_deceased"].sum(),
}
for name, value in checks.items():
    print(f"{name:28} {value}")

rows                         1163
null birthdates              0
birthdates in the future     0
death before birth           0
duplicate ids                0
deceased patients            163
